## EJERCICIO 2

In [37]:
import numpy as np
import random
import time
from tqdm import tqdm

class Hormiga:
    def __init__(self, num_ciudades):
        self.num_ciudades = num_ciudades
        self.recorrido = []
        self.longitud = 0.0

    def construir_camino(self, distancias, feromonas, alpha, beta):
        ciudad_actual = 0
        self.recorrido = [ciudad_actual]
        ciudades_pendientes = set(range(1, self.num_ciudades))
        self.longitud = 0

        while ciudades_pendientes:
            probabilidades = []
            ciudades_posibles = list(ciudades_pendientes)
            
            # p_ij = (sigma^alpha * eta^beta) / sumatoria
            for ciudad_destino in ciudades_posibles:
                tau = feromonas[ciudad_actual][ciudad_destino] ** alpha
                heuristica = 1 / distancias[ciudad_actual][ciudad_destino]
                eta = heuristica ** beta
                probabilidades.append(tau * eta)
                
            suma_prob = sum(probabilidades)

            for i, p in enumerate(probabilidades):
                probabilidades[i] = p / suma_prob
            
            # proximo nodo
            siguiente_ciudad = random.choices(ciudades_posibles, weights=probabilidades)[0]
            
            # Agregar paso al camino
            self.longitud += distancias[ciudad_actual][siguiente_ciudad]
            self.recorrido.append(siguiente_ciudad)
            ciudades_pendientes.remove(siguiente_ciudad)
            ciudad_actual = siguiente_ciudad

        # volver al origen
        self.longitud += distancias[self.recorrido[-1]][self.recorrido[0]]


class ColoniaHormigas_AS:
    def __init__(self, distancias, num_hormigas=17, iteraciones=100, rho=0.5, tau=100, alpha=1.0, beta=2.0):
        self.distancias = distancias
        self.num_ciudades = len(distancias)
        self.num_hormigas = num_hormigas
        self.iteraciones = iteraciones
        self.rho = rho
        self.tau = tau 
        self.alpha = alpha
        self.beta = beta
        
        # U(0, sigma0)
        self.feromonas = np.random.uniform(0.01, 0.1, size=(self.num_ciudades, self.num_ciudades))
        np.fill_diagonal(self.feromonas, 0)
        
        self.mejor_recorrido = None
        self.mejor_longitud = None

    def ejecutar(self, metodo_deposito):
        for t in range(self.iteraciones):
            N_hormigas = [Hormiga(self.num_ciudades) for _ in range(self.num_hormigas)]
            
            for hormiga in N_hormigas:
                hormiga.construir_camino(self.distancias, self.feromonas, self.alpha, self.beta)
                
                if self.mejor_longitud is None or hormiga.longitud < self.mejor_longitud:
                    self.mejor_longitud = hormiga.longitud
                    self.mejor_recorrido = list(hormiga.recorrido)
                    hubo_mejora = True

            # Reducir por evaporacion
            self.feromonas = (1 - self.rho) * self.feromonas
            
            # Depositar feromonas
            for hormiga in N_hormigas:
                for i in range(self.num_ciudades):
                    origen = hormiga.recorrido[i]
                    destino = hormiga.recorrido[(i + 1) % self.num_ciudades]
                    
                    if metodo_deposito == "global":
                        aporte = self.tau / hormiga.longitud
                    elif metodo_deposito == "local":
                        aporte = self.tau / self.distancias[origen][destino]
                    elif metodo_deposito == "uniforme":
                        aporte = self.tau
                        
                    self.feromonas[origen][destino] += aporte
                    # self.feromonas[destino][origen] += aporte 
            
            # las hormigas repiten el camino
            longitudes = [h.longitud for h in N_hormigas]
            if max(longitudes) == min(longitudes) and t > 10:
                print(f"\n Las hormigas repiten el mismo camino en la it={t}")
                break

        return self.mejor_recorrido, self.mejor_longitud


if __name__ == "__main__":
    matriz_d = np.loadtxt('datos/gr17.csv', delimiter=',')

    metodos = ["global", "local", "uniforme"]
    valores_rho = [0.10, 0.50, 0.80]
    print("\n| Método   | Evaporación (p) | Mejor Distancia | Tiempo (s) |")
    print("|----------|-----------------|-----------------|------------|")
    
    for metodo in metodos:
        for rho in valores_rho:
            inicio_tiempo = time.time()
            
            CH_AS = ColoniaHormigas_AS(matriz_d, num_hormigas=50, iteraciones=100, rho=rho, tau=100, alpha=1.0, beta=2.0)
            recorrido, longitud = CH_AS.ejecutar(metodo_deposito=metodo)
            
            tiempo_total = time.time() - inicio_tiempo
            print(f"| {metodo:<8} | {rho:<15} | {longitud:<15.2f} | {tiempo_total:<10.4f} |")


| Método   | Evaporación (p) | Mejor Distancia | Tiempo (s) |
|----------|-----------------|-----------------|------------|
| global   | 0.1             | 2094.00         | 0.8226     |

 Las hormigas repiten el mismo camino en la it=65
| global   | 0.5             | 2135.00         | 0.5284     |

 Las hormigas repiten el mismo camino en la it=30
| global   | 0.8             | 2152.00         | 0.2443     |
| local    | 0.1             | 2158.00         | 0.8962     |

 Las hormigas repiten el mismo camino en la it=34
| local    | 0.5             | 2094.00         | 0.3160     |

 Las hormigas repiten el mismo camino en la it=60
| local    | 0.8             | 2131.00         | 0.5045     |
| uniforme | 0.1             | 2094.00         | 0.7549     |

 Las hormigas repiten el mismo camino en la it=40
| uniforme | 0.5             | 2088.00         | 0.3073     |

 Las hormigas repiten el mismo camino en la it=27
| uniforme | 0.8             | 2158.00         | 0.2108     |
